# E-commerce Sales Analysis

## 1. Introduction and Business Objectives

This notebook analyses the performance of an online marketplace for a configurable
range of months and compares it with the period of equal length that comes right before.
By default it covers the full year 2023 and compares it with 2022.

The analysis answers these business questions:

- **Revenue:** How much revenue was generated compared with the previous period, and how did it evolve month over month?
- **Orders:** How many orders were delivered and what is the average order value?
- **Products:** Which product categories generate the most revenue?
- **Geography:** Which states generate the most revenue?
- **Customer experience:** How long do deliveries take, how satisfied are customers, and does delivery time relate to review scores?
- **Order fulfilment:** What share of orders ends in each status?

Calculations live in two modules next to this notebook: `data_loader.py` (loading, joining, filtering)
and `business_metrics.py` (metric calculations). The notebook only configures the analysis, calls
those functions and draws the charts.

## Table of Contents

1. [Introduction and Business Objectives](#1.-Introduction-and-Business-Objectives)
2. [Data Dictionary](#2.-Data-Dictionary)
3. [Data Loading and Configuration](#3.-Data-Loading-and-Configuration)
4. [Data Preparation and Transformation](#4.-Data-Preparation-and-Transformation)
5. [Business Metrics](#5.-Business-Metrics)
    - [5.1 Revenue and Orders](#5.1-Revenue-and-Orders)
    - [5.2 Product Analysis](#5.2-Product-Analysis)
    - [5.3 Geographic Analysis](#5.3-Geographic-Analysis)
    - [5.4 Customer Experience](#5.4-Customer-Experience)
6. [Summary of Observations](#6.-Summary-of-Observations)

## 2. Data Dictionary

The raw data comes from the CSV files in `ecommerce_data/`. The columns below are the ones used in this analysis.

| Table | Column | Description |
|---|---|---|
| orders | `order_id` | Unique identifier of an order |
| orders | `customer_id` | Customer who placed the order (joins to customers) |
| orders | `order_status` | Order status: delivered, shipped, processing, pending, canceled or returned |
| orders | `order_purchase_timestamp` | Date and time the order was placed. Used to assign an order to a month |
| orders | `order_delivered_customer_date` | Date and time the order reached the customer. Empty if not delivered |
| order_items | `order_item_id` | Position of the item within its order (an order can contain several items) |
| order_items | `product_id` | Product sold (joins to products) |
| order_items | `price` | Item price in the dataset's currency, excluding freight |
| order_items | `freight_value` | Shipping cost of the item |
| products | `product_category_name` | Category of the product |
| customers | `customer_state` | US state code of the customer |
| reviews | `review_score` | Customer rating of the order, from 1 (lowest) to 5 (highest) |

**Business terms used in this notebook**

| Term | Definition |
|---|---|
| Analysis period | The months selected in the configuration cell |
| Comparison period | The period of equal length immediately before the analysis period, unless set manually |
| Revenue | Sum of item `price` for delivered orders. Freight is not included |
| Order | A distinct `order_id` with delivered status |
| Average order value (AOV) | Revenue divided by the number of delivered orders |
| Month-over-month (MoM) growth | Change in monthly revenue relative to the previous calendar month |
| Delivery days | Whole days between purchase and delivery |
| Reviewed order | A delivered order that has a review score. Delivery and review metrics only use these orders |

## 3. Data Loading and Configuration

Change the values in the configuration cell to analyse a different range of months. Months are written as `"YYYY-MM"`.
Leave the comparison months as `None` to compare with the period of equal length right before the analysis period.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import plotly.express as px

import business_metrics as bm
import data_loader as dl

# ---- Analysis configuration ---------------------------------------------------
DATA_DIR = "ecommerce_data"

START_MONTH = "2023-01"          # first month of the analysis period
END_MONTH = "2023-12"            # last month of the analysis period (inclusive)

COMPARISON_START_MONTH = None    # optional, "YYYY-MM"; None = equal-length period before START_MONTH
COMPARISON_END_MONTH = None      # optional, "YYYY-MM"; set together with the start

# Delivery time groups used in section 5.4: upper edge in days of each group except the last.
DELIVERY_BINS = (3, 7)
DELIVERY_LABELS = ("1-3 days", "4-7 days", "8+ days")

# ---- Chart style ----------------------------------------------------------------
COLOR_CURRENT = "#1f4e79"        # analysis period
COLOR_PREVIOUS = "#8c9bab"       # comparison period
COLOR_SEQUENTIAL = "Blues"       # colour scale for maps
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})
pd.options.display.float_format = "{:,.2f}".format

## 4. Data Preparation and Transformation

`prepare_analysis_data` performs these steps:

1. Loads the six CSV files and parses their date columns.
2. Joins order items with orders, products, customers and review scores into one item-level table.
3. Derives the purchase month and the delivery time in days.
4. Filters the data to the analysis period and to the comparison period.
5. Creates a delivered-only version of each period, which is the basis for revenue and order metrics.

Review scores are attached to the order, so each item of an order carries that order's score.
Metrics that depend on reviews first reduce the data to one row per order.

In [ ]:
data = dl.prepare_analysis_data(
    start=START_MONTH,
    end=END_MONTH,
    data_dir=DATA_DIR,
    comparison_start=COMPARISON_START_MONTH,
    comparison_end=COMPARISON_END_MONTH,
)

sales = data["sales"]
sales_delivered = data["sales_delivered"]
sales_delivered_previous = data["sales_delivered_previous"]
orders = data["orders"]
orders_previous = data["orders_previous"]
current_start, current_end = data["periods"]["current"]
previous_start, previous_end = data["periods"]["previous"]

CURRENT_LABEL = f"{current_start.strftime('%b %Y')} to {current_end.strftime('%b %Y')}"
PREVIOUS_LABEL = f"{previous_start.strftime('%b %Y')} to {previous_end.strftime('%b %Y')}"

print(f"Analysis period:   {CURRENT_LABEL}")
print(f"Comparison period: {PREVIOUS_LABEL}")
print()
print(f"Items in analysis period (all statuses): {len(sales):,}")
print(f"Items in analysis period (delivered):    {len(sales_delivered):,}")
print(f"Items in comparison period (delivered):  {len(sales_delivered_previous):,}")

if sales_delivered.empty:
    raise ValueError("No delivered sales in the selected period. Check the configured months.")
if sales_delivered_previous.empty:
    print("Warning: no delivered sales in the comparison period; comparisons will be empty.")

In [ ]:
sales_delivered.head()

## 5. Business Metrics

All revenue and order metrics use delivered orders only. Percentage changes compare the analysis period with the comparison period.

### 5.1 Revenue and Orders

The table summarises the headline indicators for both periods. The chart shows monthly revenue; the comparison period is drawn
month by month alongside it (first month of each period against first month of the other, and so on).

In [ ]:
kpis = bm.kpi_summary(sales_delivered, sales_delivered_previous)
kpis.columns = [CURRENT_LABEL, PREVIOUS_LABEL, "Change (%)"]
kpis

In [ ]:
def plot_monthly_revenue(current, previous, current_label, previous_label):
    """Line chart of monthly revenue for the analysis period, with the comparison period overlaid.

    Both periods are aligned by month position, so the n-th month of one is drawn over the n-th month of the other.
    """
    fig, ax = plt.subplots()
    positions = range(len(current))
    ax.plot(positions, current.values, marker="o", color=COLOR_CURRENT, linewidth=2, label=current_label)
    if len(previous) > 0:
        prev_aligned = previous.reindex(
            pd.period_range(previous_start, previous_end, freq="M"), fill_value=0.0
        )
        ax.plot(range(len(prev_aligned)), prev_aligned.values, marker="o", color=COLOR_PREVIOUS,
                linewidth=2, linestyle="--", label=previous_label)
    ax.set_xticks(list(positions))
    ax.set_xticklabels([p.strftime("%b %Y") for p in current.index], rotation=45, ha="right")
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
    ax.set_xlabel("Month of purchase (comparison period aligned by month position)")
    ax.set_ylabel("Revenue (currency units)")
    ax.set_title(f"Monthly Revenue, {current_label}")
    ax.legend(title="Period")
    plt.tight_layout()
    plt.show()


current_monthly = bm.monthly_revenue(sales_delivered)
current_monthly = current_monthly.reindex(pd.period_range(current_start, current_end, freq="M"), fill_value=0.0)
plot_monthly_revenue(current_monthly, bm.monthly_revenue(sales_delivered_previous), CURRENT_LABEL, PREVIOUS_LABEL)

In [ ]:
mom_growth = bm.month_over_month_growth(sales_delivered)
mom_table = pd.DataFrame({
    "Revenue": bm.monthly_revenue(sales_delivered),
    "MoM growth (%)": 100 * mom_growth,
})
mom_table.index = mom_table.index.strftime("%b %Y")
display(mom_table)

print(f"Average month-over-month revenue growth, {CURRENT_LABEL}: "
      f"{100 * bm.average_month_over_month_growth(sales_delivered):.2f}%")

### 5.2 Product Analysis

Revenue by product category for the analysis period, ordered from highest to lowest.

In [ ]:
category_revenue = bm.revenue_by_category(sales_delivered)

fig, ax = plt.subplots()
ax.bar(category_revenue.index, category_revenue.values, color=COLOR_CURRENT)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax.set_xlabel("Product category")
ax.set_ylabel("Revenue (currency units)")
ax.set_title(f"Revenue by Product Category, {CURRENT_LABEL}")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 5.3 Geographic Analysis

Revenue by customer state for the analysis period. Darker states generate more revenue.

In [ ]:
state_revenue = bm.revenue_by_state(sales_delivered)

fig = px.choropleth(
    state_revenue,
    locations="customer_state",
    color="price",
    locationmode="USA-states",
    scope="usa",
    color_continuous_scale=COLOR_SEQUENTIAL,
    labels={"price": "Revenue (currency units)", "customer_state": "State"},
    title=f"Revenue by State, {CURRENT_LABEL}",
)
fig.show()

In [ ]:
state_revenue.head(10).rename(columns={"customer_state": "State", "price": "Revenue"})

### 5.4 Customer Experience

Delivery time and review metrics only include delivered orders that have a review score. The groups below use the
boundaries set in the configuration cell.

In [ ]:
print(f"Average delivery time, {CURRENT_LABEL}: {bm.average_delivery_days(sales_delivered):.2f} days")
print(f"Average review score,  {CURRENT_LABEL}: {bm.average_review_score(sales_delivered):.2f} out of 5")

In [ ]:
review_by_days = bm.review_score_by_delivery_days(sales_delivered)
review_by_days.columns = ["Delivery days", "Average review score", "Reviewed orders"]
review_by_days

In [ ]:
review_by_bucket = bm.review_score_by_delivery_bucket(sales_delivered, DELIVERY_BINS, DELIVERY_LABELS)

fig, ax = plt.subplots()
bars = ax.bar(review_by_bucket["delivery_bucket"].astype(str), review_by_bucket["avg_review_score"], color=COLOR_CURRENT)
ax.bar_label(bars, fmt="%.2f", padding=3)
ax.set_ylim(0, 5)
ax.set_xlabel("Delivery time (days from purchase to delivery)")
ax.set_ylabel("Average review score (1 to 5)")
ax.set_title(f"Average Review Score by Delivery Time, {CURRENT_LABEL}")
plt.tight_layout()
plt.show()

review_by_bucket.rename(columns={
    "delivery_bucket": "Delivery time", "avg_review_score": "Average review score", "orders": "Reviewed orders"
})

In [ ]:
score_share = bm.review_score_distribution(sales_delivered)

fig, ax = plt.subplots()
ax.barh(score_share.index.astype(int).astype(str), score_share.values, color=COLOR_CURRENT)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.invert_yaxis()
ax.set_xlabel("Share of reviewed orders")
ax.set_ylabel("Review score (1 = lowest, 5 = highest)")
ax.set_title(f"Distribution of Review Scores, {CURRENT_LABEL}")
plt.tight_layout()
plt.show()

In [ ]:
status_share = bm.order_status_distribution(orders)

fig, ax = plt.subplots()
ax.barh(status_share.index, status_share.values, color=COLOR_CURRENT)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
ax.invert_yaxis()
ax.set_xlabel("Share of all orders placed in the period")
ax.set_ylabel("Order status")
ax.set_title(f"Order Status Distribution, {CURRENT_LABEL}")
plt.tight_layout()
plt.show()

## 6. Summary of Observations

The text below is generated from the configured periods, so it stays correct when the date range changes.
It reports values and direction of change only; interpretation against targets is left to the reader.

In [ ]:
def describe_change(name, current, previous, unit="", decimals=1):
    """Return one sentence describing how a metric moved versus the comparison period."""
    change = bm.growth_rate(current, previous)
    if pd.isna(change):
        return f"{name}: {current:,.{decimals}f}{unit} (no comparison available)."
    direction = "up" if change > 0 else "down" if change < 0 else "unchanged"
    return (f"{name}: {current:,.{decimals}f}{unit}, {direction} {abs(100 * change):.1f}% "
            f"versus {previous:,.{decimals}f}{unit} in {PREVIOUS_LABEL}." if direction != "unchanged"
            else f"{name}: {current:,.{decimals}f}{unit}, unchanged versus {PREVIOUS_LABEL}.")


top_category = category_revenue.index[0]
top_category_share = category_revenue.iloc[0] / category_revenue.sum()
top_state = state_revenue.iloc[0]
top_state_share = top_state["price"] / state_revenue["price"].sum()
delivered_share = status_share.get("delivered", 0.0)

print(f"Observations for {CURRENT_LABEL} (compared with {PREVIOUS_LABEL})")
print()
print("Revenue and orders")
print("- " + describe_change("Total revenue", kpis.loc["Total revenue"].iloc[0], kpis.loc["Total revenue"].iloc[1], decimals=0))
print("- " + describe_change("Delivered orders", kpis.loc["Total orders"].iloc[0], kpis.loc["Total orders"].iloc[1], decimals=0))
print("- " + describe_change("Average order value", kpis.loc["Average order value"].iloc[0], kpis.loc["Average order value"].iloc[1], decimals=2))
print(f"- Average month-over-month revenue growth: {100 * bm.average_month_over_month_growth(sales_delivered):.2f}%")
best_month = current_monthly.idxmax()
worst_month = current_monthly.idxmin()
print(f"- Highest revenue month: {best_month.strftime('%b %Y')} ({current_monthly.max():,.0f}); lowest: {worst_month.strftime('%b %Y')} ({current_monthly.min():,.0f})")
print()
print("Products and geography")
print(f"- Top category: {top_category}, {100 * top_category_share:.1f}% of revenue")
print(f"- Top state: {top_state['customer_state']}, {100 * top_state_share:.1f}% of revenue")
print()
print("Customer experience")
print("- " + describe_change("Average review score", kpis.loc["Average review score"].iloc[0], kpis.loc["Average review score"].iloc[1], decimals=2))
print("- " + describe_change("Average delivery days", kpis.loc["Average delivery days"].iloc[0], kpis.loc["Average delivery days"].iloc[1], decimals=2))
for _, row in review_by_bucket.iterrows():
    print(f"- Delivery time {row['delivery_bucket']}: average review score {row['avg_review_score']:.2f} ({int(row['orders']):,} reviewed orders)")
print(f"- Share of orders placed in the period that were delivered: {100 * delivered_share:.1f}%")